# 02 Event Study

Purpose: explain whether event-window returns came from company-specific reaction, QQQ beta, or semiconductor sector beta.

Before running this notebook, import the curated event loop files and run:

```bash
uv run python -m scripts.import_events data/manual/events_ai_compute.csv
uv run python -m scripts.import_event_impacts data/manual/event_impacts_ai_compute.csv
uv run python -m scripts.import_event_metrics data/manual/event_metrics_ai_compute.csv
uv run python -m scripts.build_event_returns --benchmarks QQQ SOXX SMH
uv run python -m scripts.build_event_reviews
```

In [ ]:
import duckdb

DB = '../data/duckdb/quant_learn.duckdb'
con = duckdb.connect(DB, read_only=True)

In [ ]:
events = con.execute('''
select e.event_date, e.reaction_date, r.affected_ticker, e.event_type, e.event_name,
       r.return_window, r.benchmark_ticker,
       r.raw_return, r.benchmark_return, r.abnormal_return,
       r.data_quality_flag, r.missing_reason,
       m.metric_name, m.actual_value, m.expected_value, m.surprise_pct
from event_returns r
join events e using (event_id)
left join event_metrics m using (event_id)
order by e.event_date desc, r.affected_ticker, r.return_window, r.benchmark_ticker
''').fetchdf()
events

In [ ]:
group_cols = ['event_type', 'affected_ticker', 'return_window', 'benchmark_ticker']
summary = events.groupby(group_cols).agg(
    n=('event_type', 'size'),
    avg_raw_return=('raw_return', 'mean'),
    avg_abnormal_return=('abnormal_return', 'mean'),
).reset_index()
summary

In [ ]:
reviews = con.execute('''
select event_id, affected_ticker, reaction_date, event_type,
       data_quality_flag, confidence,
       raw_reaction_summary, benchmark_attribution_summary,
       metric_surprise_summary, interpretation, thesis_impact
from event_reviews
order by reaction_date desc, event_id, affected_ticker
''').fetchdf()
reviews

Five-sentence event review:

1. What happened?
2. What did the market likely expect beforehand?
3. Which actual metric surprised: revenue, segment growth, margin, CapEx, guidance, or risk?
4. Was the price reaction company alpha, QQQ beta, or SOXX/SMH sector beta?
5. Did this event change the next 1-2 quarter research hypothesis?